# Fundamentals of Accelerator Physics and Technology
### Simulations and Measurements Lab
# Computer Lab: Simulated beam transport with quadrupole focusing — Xsuite version
##### Original authors: K. Ruisard, N. Evans and V. S. Morozov  
##### Local Xsuite remake

This notebook replaces the original Sirepo/Elegant workflow with local Python and Xsuite. Questions for students are still numbered.

The front-facing code cells expose the physics knobs that students should be able to change. Plotting, dense line slicing, matching optimization, and particle-distribution generation are kept in the helper module `uspas_labs.quadrupole_focusing`, installed from the course GitHub repo.

<style>
.answer {
    color: #b00020;
    border-left: 4px solid #b00020;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.answer table, .answer th, .answer td {
    color: #b00020;
}
.instructor-note {
    color: #7a3b00;
    border-left: 4px solid #7a3b00;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.note {
    background-color: #f6f8fa;
    padding: 0.75em;
    border-left: 4px solid #999;
}
</style>

## 1. Setup

At injection, or at the start of a simulation, there is an optimal spot size and angular divergence for the beam. This is the **matched condition**. In a periodic focusing structure, such as a FODO line, the matched Twiss solution is periodic with the lattice period.

We describe the matched envelope with Twiss parameters $\beta_x$, $\beta_y$, $\alpha_x$, and $\alpha_y$. The RMS beam size is related to beta and geometric emittance by

$$
\sigma_x = \sqrt{\beta_x\epsilon_x}, \qquad \sigma_y = \sqrt{\beta_y\epsilon_y}.
$$

| Parameter | Value |
|---|---:|
| Species | Electron |
| Energy | 1 GeV |
| $\epsilon_x$ | $6\ \mathrm{mm\,mrad}$ |
| $\epsilon_y$ | $6\ \mathrm{mm\,mrad}$ |
| FODO period | $5.0\ \mathrm{m}$ |
| Quadrupole-center spacing | $2.5\ \mathrm{m}$ |
| Quadrupole geometric strength | $K_1 = 0.6\ \mathrm{m}^{-2}$ |
| Quadrupole length | $0.5\ \mathrm{m}$ |

The default FODO sequence starts halfway through a drift between quadrupoles:

`drift — focusing quadrupole — drift — defocusing quadrupole — drift`.


In [ ]:
HELPER_VERSION = "main"  # Replace with a release tag, e.g. "v2026-lab1", for student releases.
HELPER_REPO = "git+https://github.com/r-hensley/uspas-2026-jupyter.git"

import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--upgrade",
    f"{HELPER_REPO}@{HELPER_VERSION}",
])

import numpy as np
import pandas as pd
import xtrack as xt
from IPython.display import display

from uspas_labs import quadrupole_focusing as qfh

try:
    from google.colab import output as colab_output
except ImportError:
    qfh.set_plotly_renderer("plotly_mimetype")
else:
    colab_output.enable_custom_widget_manager()
    qfh.set_plotly_renderer("colab")

qfh.print_versions()


## 2. Beamline matching

### A) Unmatched beam

Initially, propagate a deliberately unmatched beam through one FODO cell. Use

$$
\beta_x=\beta_y=4\ \mathrm{m}, \qquad \alpha_x=\alpha_y=0.
$$

The cell below builds the FODO line using the modern Xsuite `Environment` pattern: define named elements with `env.new(...)`, assemble a line with `env.new_line(...)`, assign a reference particle, and call `line.twiss(...)`.

Run the cell below and compare the beta functions with the RMS beam sizes.

In [ ]:
env = xt.Environment()
env["k1_fodo"] = 0.6

env.new("D1", xt.Drift, length=1.0)
env.new("QF", xt.Quadrupole, length=0.5, k1="k1_fodo")
env.new("D2", xt.Drift, length=2.0)
env.new("QD", xt.Quadrupole, length=0.5, k1="-k1_fodo")
env.new("D3", xt.Drift, length=1.0)

fodo_cell = env.new_line(
    name="fodo_cell",
    components=["D1", "QF", "D2", "QD", "D3"],
)
fodo_cell.particle_ref = xt.Particles(
    p0c=1e9,
    mass0=xt.ELECTRON_MASS_EV,
    q0=-1,
)
fodo_cell.build_tracker()

display(fodo_cell.get_table().to_pandas()[["name", "s_start", "s_end", "element_type"]])

tw_unmatched_cell = fodo_cell.twiss(
    method="4d",
    betx=4.0, alfx=0.0,
    bety=4.0, alfy=0.0,
)
tw_unmatched_dense = qfh.twiss_dense(
    fodo_cell,
    betx=4.0, alfx=0.0,
    bety=4.0, alfy=0.0,
)

qfh.plot_beta_and_sigma(tw_unmatched_dense, "Unmatched Twiss functions and RMS sizes through one FODO cell")
qfh.twiss_dataframe(tw_unmatched_cell)

### B) Solving for the matched solution

Now ask Xsuite for the periodic one-cell Twiss solution. In the original Sirepo/Elegant lab this was done by setting `Matched = Yes`.

From the matched lattice function, the phase advance is

$$
\psi_x = \int \frac{ds}{\beta_x(s)}, \qquad \psi_y = \int \frac{ds}{\beta_y(s)}.
$$

Xsuite reports `qx` and `qy` as tune-like phase advances in **turns per pass**. For this one-cell line, one pass is one FODO cell. Multiply by $2\pi$ to get radians.

**Q0) Calculate the X and Y phase advances for the single FODO cell.**


In [ ]:
tw_cell = fodo_cell.twiss(method="4d")
tw_cell_dense = qfh.twiss_dense(fodo_cell)

qfh.plot_beta_and_sigma(tw_cell_dense, "Matched Twiss functions and RMS sizes for one FODO cell")
qfh.phase_advance_table(tw_cell)

### Try it: change the FODO strength

Move the sliders and watch how the beta functions, beam size, and phase advance change. This reproduces the useful interactivity of the old Sirepo version while keeping the Xsuite workflow visible.


In [ ]:
qfh.interactive_fodo_strength()


The thin-lens approximation gives the following FODO estimates:

$$
\beta_{\max}=L \frac{1 + \sin( \psi/2)}{\sin \psi},
\qquad
\beta_{\min}=L \frac{1 - \sin(\psi/2)}{\sin \psi}.
$$

<div class="note">
The original worksheet states that the half-cell length is $2.5\ \mathrm{m}$, but the thin-lens comparison described as “quite close” is obtained by using the full $5.0\ \mathrm{m}$ FODO period in the formula above. The code below makes that convention explicit as `length_for_formula=5.0`.
</div>

**Q1) For this cell, calculate $\beta_{\min}$ and $\beta_{\max}$ in two ways:**

- A) thin-lens prediction
- B) using Xsuite

**Q2) If you increased the length of the quadrupole elements while holding both the cell length and the phase advance $\psi$ fixed, would the difference between the thin-lens and Xsuite results get larger or smaller? Explain.**


In [ ]:
qfh.thin_lens_comparison_table(tw_cell_dense, length_for_formula=5.0)


In [ ]:
quad_length_scan = qfh.quad_length_fixed_phase_table(target_q=tw_cell.qx)
quad_length_scan


### Try it: test the thick-lens effect directly

This slider changes the quadrupole length. The helper then adjusts $k_1$ so that the one-cell phase advance remains the same as the default cell. Watch the thick-lens beta extrema move away from the fixed thin-lens prediction.


In [ ]:
qfh.interactive_quad_length_effect(target_q=tw_cell.qx)


**Q3) What are the average, maximum, and minimum RMS beam spot sizes for a matched beam in this lattice?**

Use $\sigma=\sqrt{\beta\epsilon}$ with $\epsilon_x=\epsilon_y=6\ \mathrm{mm\,mrad}$.


In [ ]:
qfh.beam_size_summary(tw_cell_dense, label="matched FODO cell")


**Q4) Where along the lattice is the beam round? Where is it largest and smallest in the horizontal plane? In the vertical plane? Give your answer in $s$ and in terms of elements or regions.**


In [ ]:
qfh.location_table(tw_cell_dense)


### C) Matched beam propagation down a 100 m FODO beamline

Now repeat the same FODO cell 20 times. The total line length is $20\times5\ \mathrm{m}=100\ \mathrm{m}$.

**Q5) Confirm that the tune of this 20-cell lattice is consistent with the one-cell solution.**


In [ ]:
fodo_100m = qfh.make_fodo_line(n_cells=20, k1=0.6)
tw_100m = qfh.twiss_periodic(fodo_100m)
tw_100m_dense = qfh.twiss_dense(fodo_100m, points_per_meter=30)

qfh.plot_sigmas(tw_100m_dense, "Matched RMS beam sizes through 20 FODO cells")

pd.DataFrame({
    "quantity": ["nu_x over 100 m", "nu_y over 100 m", "psi_x over 100 m [rad]", "psi_y over 100 m [rad]", "nu_x per cell", "nu_y per cell", "psi_x per cell [rad]", "psi_y per cell [rad]"],
    "value": [tw_100m.qx, tw_100m.qy, 2*np.pi*tw_100m.qx, 2*np.pi*tw_100m.qy, tw_100m.qx/20, tw_100m.qy/20, 2*np.pi*tw_100m.qx/20, 2*np.pi*tw_100m.qy/20],
})


**Q6) How many oscillations does a single particle make in 100 m?**

You can calculate this from the tune, but the cell below also launches a 1 mm horizontal centroid offset. The centroid follows the same linear equations as a single particle.


In [ ]:
tw_centroid = qfh.centroid_orbit(fodo_100m, x0=1e-3, px0=0.0)
qfh.plot_centroid(tw_centroid)


### D) Propagation of a mismatched beam

Initialize the beam with a 10% beta mismatch relative to the matched solution:

$$
\beta_x = 1.1\beta_{x,\mathrm{matched}},\qquad
\beta_y = 1.1\beta_{y,\mathrm{matched}},
$$

while keeping the matched alpha values.

**Q7) Count the approximate number of envelope mismatch oscillations over the 100 m beamline. Does it agree with the smooth-approximation prediction $\nu_{\mathrm{envelope}}=2\nu$?**


In [ ]:
tw_mismatch_dense = qfh.twiss_dense(
    fodo_100m,
    points_per_meter=30,
    betx=float(tw_cell.betx[0]) * 1.10,
    alfx=float(tw_cell.alfx[0]),
    bety=float(tw_cell.bety[0]) * 1.10,
    alfy=float(tw_cell.alfy[0]),
)

qfh.plot_mismatch(tw_100m_dense, tw_mismatch_dense, "Matched and 10% mismatched RMS beam envelopes")
qfh.plot_cell_boundary_envelope(tw_mismatch_dense)
qfh.cell_start_envelope_table(tw_mismatch_dense).head(10)


### Try it: change the injection mismatch

The slider changes the initial beta mismatch factor. Values closer to 1 are closer to matched; values farther from 1 produce stronger beating.


In [ ]:
qfh.interactive_mismatch(tw_cell)


### E) Matched solution with decreased quadrupole strength

Now reduce the quadrupole strength while keeping the emittance fixed. The RMS envelope equation contains a focusing term and an emittance-pressure term,

$$
\frac{d^2 \sigma_x}{ds^2}=-K_x\sigma_x+\frac{\epsilon_x^2}{\sigma_x^3}.
$$

With weaker quadrupoles, the focusing term is weaker relative to the emittance term.

**Q8) For a new FODO cell with $|k_1|=0.2\ \mathrm{m}^{-2}$, record the matched beam-size statistics and phase advance.**


In [ ]:
fodo_cell_weak = qfh.make_fodo_line(n_cells=1, k1=0.2)
tw_cell_weak = qfh.twiss_periodic(fodo_cell_weak)
tw_cell_weak_dense = qfh.twiss_dense(fodo_cell_weak)

qfh.plot_beta_and_sigma(tw_cell_weak_dense, "Matched weak FODO cell: |k1| = 0.2 1/m^2")
display(qfh.beam_size_summary(tw_cell_weak_dense, label="weak FODO cell"))
display(qfh.phase_advance_table(tw_cell_weak))


### Try it: vary the matched cell strength

Use this slider to connect the strong-cell and weak-cell limits continuously.


In [ ]:
qfh.interactive_weak_cell()


### F) Injection mismatch between different FODO sections

Build a hybrid beamline containing 10 cells with $|k_1|=0.6\ \mathrm{m}^{-2}$ followed by 10 cells with $|k_1|=0.5\ \mathrm{m}^{-2}$. Start the line with the matched Twiss parameters of the strong cell.

**Q9) Describe what happens as the beam transitions from the stronger to the weaker FODO cell. Is the beam matched in the first 10 cells? What about the last 10 cells?**


In [ ]:
hybrid_line = qfh.make_hybrid_fodo_line(first_cells=10, second_cells=10, k1_first=0.6, k1_second=0.5)

tw_hybrid_from_strong_match = qfh.twiss_dense(
    hybrid_line,
    points_per_meter=30,
    betx=float(tw_cell.betx[0]), alfx=float(tw_cell.alfx[0]),
    bety=float(tw_cell.bety[0]), alfy=float(tw_cell.alfy[0]),
)

qfh.plot_hybrid_transition(tw_hybrid_from_strong_match, transition_s=50.0)
qfh.optics_summary(tw_hybrid_from_strong_match, label="strong-cell initial match")


### Try it: change the downstream focusing strength

The first section remains fixed. Change the second-section strength and observe the mismatch at the transition.


In [ ]:
qfh.interactive_hybrid_transition(tw_cell)


Now ask Xsuite for the periodic solution of the entire 100 m hybrid line.

**Q10) Is this new solution matched over the period $L=100\ \mathrm{m}$? Is it matched for the period $L=5\ \mathrm{m}$? Does there exist one solution that is matched for period $L=5\ \mathrm{m}$ in all 20 cells?**


In [ ]:
tw_hybrid_periodic = qfh.twiss_periodic(hybrid_line)
tw_hybrid_periodic_dense = qfh.twiss_dense(hybrid_line, points_per_meter=30)

qfh.plot_hybrid_transition(tw_hybrid_periodic_dense, transition_s=50.0, title="Periodic solution of the full 100 m hybrid line")

pd.DataFrame({
    "quantity": ["beta_x start [m]", "beta_x end [m]", "alpha_x start", "alpha_x end", "Qx over 100 m", "Qy over 100 m"],
    "value": [tw_hybrid_periodic.betx[0], tw_hybrid_periodic.betx[-1], tw_hybrid_periodic.alfx[0], tw_hybrid_periodic.alfx[-1], tw_hybrid_periodic.qx, tw_hybrid_periodic.qy],
})


### G) Matching-section insert

A real accelerator contains insertions, injection and extraction regions, experiments, and other sections that are not just repeated FODO cells. A **matching section** uses adjustable quadrupoles to transform the incoming Twiss parameters into the desired outgoing Twiss parameters.

Here the incoming beam is taken from the end of the matched FODO cell. The target at the end of the matching section is a round, uncorrelated distribution:

$$
\beta_x=\beta_y, \qquad \alpha_x=0, \qquad \alpha_y=0.
$$

The section has four quadrupoles. This time the notebook shows both the Xsuite `Environment` construction and the `line.match(...)` call directly. The helper functions only summarize and plot the result.

In [ ]:
initial_from_fodo_end = qfh.fodo_end_twiss_dict(tw_cell)

match_env = xt.Environment()
match_env["k1_fodo"] = env["k1_fodo"]
match_env.new("DM0", xt.Drift, length=1.0)

match_components = ["DM0"]
for i, sign in enumerate([+1, -1, +1, -1], start=1):
    match_env[f"kQM{i}"] = sign * match_env["k1_fodo"]
    match_env.new(f"QM{i}", xt.Quadrupole, length=0.5, k1=f"kQM{i}")
    match_env.new(f"DM{i}", xt.Drift, length=2.0)
    match_components.extend([f"QM{i}", f"DM{i}"])

match_section = match_env.new_line(name="match_section", components=match_components)
match_section.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)
match_section.build_tracker()

display(match_section.get_table().to_pandas()[["name", "s_start", "s_end", "element_type"]])

tw_match_before = match_section.twiss(method="4d", **initial_from_fodo_end)
tw_match_before_dense = qfh.twiss_dense(match_section, points_per_meter=80, **initial_from_fodo_end)

opt = match_section.match(
    method="4d",
    vary=[
        xt.Vary("kQM1", step=1e-3, limits=(-3.0, 3.0)),
        xt.Vary("kQM2", step=1e-3, limits=(-3.0, 3.0)),
        xt.Vary("kQM3", step=1e-3, limits=(-3.0, 3.0)),
        xt.Vary("kQM4", step=1e-3, limits=(-3.0, 3.0)),
    ],
    targets=[
        xt.Target("alfx", 0.0, at="_end_point", tol=1e-8),
        xt.Target("alfy", 0.0, at="_end_point", tol=1e-8),
        xt.Target(
            lambda tw: tw["betx", "_end_point"] - tw["bety", "_end_point"],
            0.0,
            tol=1e-8,
            tag="betx_minus_bety",
        ),
    ],
    n_steps_max=80,
    **initial_from_fodo_end,
)

tw_match_after = match_section.twiss(method="4d", **initial_from_fodo_end)
tw_match_after_dense = qfh.twiss_dense(match_section, points_per_meter=80, **initial_from_fodo_end)

opt.target_status()
qfh.plot_matching_comparison(tw_match_before_dense, tw_match_after_dense)
qfh.matching_summary(match_section, tw_match_after)

### Try it: match by hand

Before relying on the optimizer, adjust the four quadrupole knobs manually. The endpoint target is $\beta_x-\beta_y\to0$, $\alpha_x\to0$, and $\alpha_y\to0$.


In [ ]:
qfh.interactive_manual_match(initial_from_fodo_end)


**Q11) Attach graphs of the input and output $xx'$ distributions.**

**Q12) Attach graphs of $\beta_{x,y}$ before and after optimization.**

**Q13) What are the final values of $\beta_{x,y}$ at the end of `MATCHsection`?**

**Q14) What are the optimized strengths of the quadrupoles in `MATCHsection`?**

**Q15) Extra credit: design an entire injection insertion and show the resulting optics.**


In [ ]:
tracked_distribution = qfh.track_distribution(match_section, initial_from_fodo_end, n_particles=2500, seed=11)
qfh.plot_tracked_xpx_distribution(tracked_distribution, "Tracked input/output horizontal phase-space distributions")
qfh.plot_phase_space_ellipses(tw_match_before, tw_match_after)


In [ ]:
insertion_line = qfh.make_injection_insertion_line(fodo_cell, match_section)
tw_insertion_dense = qfh.twiss_dense(insertion_line, points_per_meter=60, **initial_from_fodo_end)
qfh.plot_twiss(tw_insertion_dense, "Example FODO + matching section + reversed matching section insertion")
